# 第4章: 言語解析

問題30から問題35までは、以下の文章`text`（太宰治の『走れメロス』の冒頭部分）に対して、言語解析を実施せよ。問題36から問題39までは、国家を説明した文書群（日本語版ウィキペディア記事から抽出したテキスト群）をコーパスとして、言語解析を実施せよ。

In [109]:
text = """
メロスは激怒した。
必ず、かの邪智暴虐の王を除かなければならぬと決意した。
メロスには政治がわからぬ。
メロスは、村の牧人である。
笛を吹き、羊と遊んで暮して来た。
けれども邪悪に対しては、人一倍に敏感であった。
"""

## 30. 動詞
文章`text`に含まれる動詞をすべて表示せよ。

In [110]:
!pip install janome

In [111]:
from janome.tokenizer import Tokenizer

t = Tokenizer()

# 文章を単語ごとに分割して品詞を確認する
for token in t.tokenize(text):
  res = token.part_of_speech.split(",")

  if (res[0] == "動詞"):
    print(token.surface)

し
除か
なら
し
わから
吹き
遊ん
暮し
来


## 31. 動詞の原型
文章`text`に含まれる動詞と、その原型をすべて表示せよ。

In [112]:
for token in t.tokenize(text):
  res = token.part_of_speech.split(",")

  if (res[0] == "動詞"):
    print(token.base_form)

する
除く
なる
する
わかる
吹く
遊ぶ
暮す
来る


## 32. 「AのB」
文章`text`において、2つの名詞が「の」で連結されている名詞句をすべて抽出せよ。

In [113]:
data = t.tokenize(text)
res = []

for i,token in enumerate(data):
  part_of_speech = token.part_of_speech.split(",")[0]
  res.append([token.surface, part_of_speech])

  if (len(res) >= 3):

    if (res[1][0] == "の"):
      if (res[0][1] == "名詞") and (res[2][1] == "名詞"):
        print(res[0][0])
        print(res[2][0])

    del res[0]

暴虐
王
村
牧人


## 33. 係り受け解析

文章`text`に係り受け解析を適用し、係り元と係り先のトークン（形態素や文節などの単位）をタブ区切り形式ですべて抽出せよ。

In [114]:
!pip install -U ginza ja-ginza

In [115]:
import spacy

# エラーを回避するための設定（config）を定義
config = {
    "components": {
        "compound_splitter": {
            "split_mode": "C",
        }
    }
}

# configを渡してGiNZA（日本語モデル）を読み込む
nlp = spacy.load("ja_ginza", config=config)

In [116]:
# 2. 解析したいテキストを渡す
doc = nlp(text.replace("\n", ""))

# 3. 係り受けの結果を1単語ずつ取り出して表示する
for token in doc:
    # token.text      : その単語（係り元）
    # token.head.text : どこにかかっているか（係り先）
    # token.dep_      : どういう関係か（主語、目的語など）
    print(f"{token.text} -> {token.head.text} ({token.dep_})")

メロス -> 激怒 (nsubj)
は -> メロス (case)
激怒 -> 激怒 (ROOT)
し -> 激怒 (aux)
た -> 激怒 (aux)
。 -> 激怒 (punct)
必ず -> 除か (advmod)
、 -> 必ず (punct)
かの -> 暴虐 (det)
邪智 -> 暴虐 (compound)
暴虐 -> 王 (nmod)
の -> 暴虐 (case)
王 -> 除か (obj)
を -> 王 (case)
除か -> 決意 (advcl)
なけれ -> 除か (aux)
ば -> なけれ (fixed)
なら -> なけれ (fixed)
ぬ -> なけれ (fixed)
と -> 除か (case)
決意 -> 決意 (ROOT)
し -> 決意 (aux)
た -> 決意 (aux)
。 -> 決意 (punct)
メロス -> わから (obl)
に -> メロス (case)
は -> メロス (case)
政治 -> わから (nsubj)
が -> 政治 (case)
わから -> わから (ROOT)
ぬ -> わから (aux)
。 -> わから (punct)
メロス -> 牧人 (nsubj)
は -> メロス (case)
、 -> メロス (punct)
村 -> 牧人 (nmod)
の -> 村 (case)
牧人 -> 牧人 (ROOT)
で -> 牧人 (cop)
ある -> で (fixed)
。 -> 牧人 (punct)
笛 -> 吹き (obj)
を -> 笛 (case)
吹き -> 暮し (advcl)
、 -> 吹き (punct)
羊 -> 遊ん (obl)
と -> 羊 (case)
遊ん -> 暮し (advcl)
で -> 遊ん (mark)
暮し -> 暮し (ROOT)
て -> 暮し (mark)
来 -> て (fixed)
た -> 暮し (aux)
。 -> 暮し (punct)
けれど -> 敏感 (cc)
も -> けれど (fixed)
邪悪 -> 敏感 (obl)
に -> 邪悪 (case)
対し -> に (fixed)
ては -> に (fixed)
、 -> 邪悪 (punct)
人 -> 倍 (compound)
一 -> 倍 (nummod)
倍 ->

## 34. 主述の関係
文章`text`において、「メロス」が主語であるときの述語を抽出せよ。

In [117]:
# 2. 解析したいテキストを渡す
doc = nlp(text.replace("\n", ""))

# 3. 係り受けの結果を1単語ずつ取り出して表示する
for token in doc:
    if (token.dep_ == "nsubj") and (token.text == "メロス"):
      print(token.head.text)

激怒
牧人


## 35. 係り受け木
「メロスは激怒した。」の係り受け木を可視化せよ。

In [118]:
from spacy import displacy

text = "メロスは激怒した。"
doc = nlp(text)

displacy.render(doc,
                style="dep"   # 描画スタイルの指定　dep -> 係り受け
                )

## 36. 単語の出現頻度

問題36から39までは、Wikipediaの記事を以下のフォーマットで書き出したファイル[jawiki-country.json.gz](/data/jawiki-country.json.gz)をコーパスと見なし、統計的な分析を行う。

* 1行に1記事の情報がJSON形式で格納される
* 各行には記事名が"title"キーに、記事本文が"text"キーの辞書オブジェクトに格納され、そのオブジェクトがJSON形式で書き出される
* ファイル全体はgzipで圧縮される

まず、第3章の処理内容を参考に、Wikipedia記事からマークアップを除去し、各記事のテキストを抽出せよ。そして、コーパスにおける単語（形態素）の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

In [166]:
!pip install mecab-python3 unidic-lite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 21.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 19.5 MB/s eta 0:00:00
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=bbf91b6de062ebef898e4cced2bbd5ea478e442b5ea696968a7d872f993d4b8e
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built unidic-lite


In [167]:
import re
import json

import MeCab
from collections import Counter

テキストを成型する関数

In [164]:
def Replace_highlight(text):
  res = re.sub(r"^[\|\!].*", "", text)
  res = re.sub(r"^\{\{基礎情報.*\}\}", "", res)

  # 強調マークアップの除去
  # 2～5個連続している ' を消す
  res = re.sub(r"'{2,5}", "", res)

  # 見出しの消去
  res = re.sub(r"={2,5}", "", res)

  # カテゴリの消去
  res = re.sub(r"\[\[Category\:[^\]]*?\]\]", "", res)

  # 言語
  res = re.sub(r"\{\{[^\{]*[lL]ang\|.+\|(.*?)\}\}", r"\1", res)
  res = re.sub(r"\{\{[lL]ang[^|]*\|(.*?)\}\}", r"\1", res)

  # {{仮リンク|エジプトの首相|en|Prime Minister of Egypt|label=首相}}
  # 仮リンクの除去
  res = re.sub(r"\{\{仮リンク\|([^\|]*)\|.*?\}\}", r"\1", res)

  #例:[[合同法 (1707年)|1707年合同法]]
  # [[から|までを消す
  # ただし、その間の文字は]以外
  res = re.sub(r"\[\[[^\]]*\|([^\]]*)\]\]", r"\1", res)



  # # 例:
  res = re.sub(r"\{\{.*?\}\}", "", res)
  res = re.sub(r"\{\|.*?\|\}", "", res)



  # # "[["と"]]"を消す
  res = re.sub(r"\[\[\#?|\]\]", "", res)

  # コメントを消す
  res = re.sub(r"<!--.*?-->", "", res)


  # MediaWikiマークアップの除去2
  # <ref>や<ref name= ……>などを消す
  # <ref (>以外の0個以上の文字)>(任意の0個以上の文字)0個か1個の</ref>
  res = re.sub(r"<ref[^>]*>.*?</ref>", "", res)

  # <br />を消す
  res = re.sub(r"<br ?/?>", "", res)
  res = re.sub(r"^\*+|;", "", res)

  # 外部リンクを消す
  res = re.sub(r"\[http[^\]]*?\]", "", res)

  return res.strip()

形態素解析する関数

In [194]:
def Morphological_analysis(text, word_counts):

  # 1. MeCabのインスタンス作成
  # 空文字を指定するとデフォルトの辞書が使われます
  tagger = MeCab.Tagger()

  # 解析結果（単語の基本形）を格納するリスト
  words = []

  # 2. 形態素解析を実行
  node = tagger.parseToNode(text)

  while node:
      # node.surface は単語の実際の文字（例: "走っ"）
      # node.feature は品詞や活用などの情報がカンマ区切りになった文字列
      features = node.feature.split(',')

      # 最初の要素が品詞
      pos = features[0]


      # 基本形（辞書形）を取得。辞書によって位置が異なる場合がありますが、
      # 一般的な辞書（ipadic等）では6番目（インデックス6）、unidic等では7番目（インデックス7）以降にあります。
      # ここでは要素数が十分にあるか確認してから取得します。
      if len(features) > 7:
          base_form = features[7] # unidic-liteの場合はこちらが多い
      elif len(features) > 6:
          base_form = features[6] # ipadicの場合
      else:
          base_form = node.surface

      # 基本形が取得できなかった場合（'*'になることがある）は、元の文字を使う
      if base_form == '*':
          base_form = node.surface

      words.append(base_form)

      node = node.next

  # 3. 単語の出現頻度をカウント
  res = Counter(words)
  word_counts.update(res)


  return word_counts

  # 4. 出現頻度の高い上位20語を取得
  top_20 = word_counts.most_common(20)

  # 5. 結果の表示
  print("=== 出現頻度 上位20語 ===")
  for rank, (word, count) in enumerate(top_20, 1):
      print(f"第{rank}位: {word} ({count}回)")

In [198]:
word_counts = Counter()

with open("jawiki-country.json", 'r') as f:
    for line in f:
      line = json.loads(line)   # 辞書型になる keyは"title", "text"

      counta = 0
      sentence = ""

      for word in line["text"].split("\n"):
        sentence += word.strip()
        counta += word.count("{")
        counta -= word.count("}")

        if (counta == 0):
          sentence = Replace_highlight(sentence)
          if (sentence):
            print(sentence)
            word_counts = Morphological_analysis(sentence, word_counts)
            sentence = ""

print(word_counts.most_common(20))

エジプト・アラブ共和国（エジプト・アラブきょうわこく、جمهورية مصر العربية）、通称エジプトは、中東（アラブ世界）および北アフリカにある共和国。首都はカイロ。
アフリカ大陸では北東端に位置し、西にリビア、南にスーダン、北東のシナイ半島ではイスラエル、ガザ地区と国境を接する。北は地中海、東は紅海に面している。南北に流れるナイル川の河谷とデルタ地帯（ナイル・デルタ）のほかは、国土の大部分の95%以上が砂漠である。ナイル河口の東に地中海と紅海を結ぶスエズ運河がある。
国号
正式名称はアラビア語で Ⲭⲏⲙⲓ（Khemi ケーミ）。
アラビア語の名称ミスルは、古代からセム語でこの地を指した名称である。なお、セム語の一派であるヘブライ語では、双数形のミスライム（מצרים, ミツライム）となる。
公式の英語表記は Arab Republic of Egypt。通称 Egypt 。形容詞はEgyptian 。エジプトの呼称は、古代エジプト語のフート・カア・プタハ（プタハ神の魂の神殿）から転じてこの地を指すようになったギリシャ語の単語である、ギリシャ神話のアイギュプトスにちなむ。
日本語の表記はエジプト・アラブ共和国。通称エジプト。漢字では埃及と表記し、埃と略す。この漢字表記は、漢文がそのまま日本語や中国語などに輸入されたものである。英語では「イージプト」と呼ばれる。
1882年 - 1922年 （イギリス領エジプト）
1922年 - 1953年 エジプト王国
1953年 - 1958年 エジプト共和国
1958年 - 1971年 アラブ連合共和国
1971年 - 現在 エジプト・アラブ共和国
歴史
古代エジプト
ギザの三大ピラミッド
ヒエログリフ
「エジプトはナイルの賜物」という古代ギリシアの歴史家ヘロドトスの言葉で有名なように、エジプトは豊かなナイル川のデルタに支えられ古代エジプト文明を発展させてきた。エジプト人は紀元前3000年ごろには早くも中央集権国家を形成し、ピラミッドや王家の谷、ヒエログリフなどを通じて世界的によく知られている高度な文明を発達させた。
アケメネス朝ペルシア
3000年にわたる諸王朝の盛衰の末、紀元前525年にアケメネス朝ペルシアに支配された。
ヘレニズム文化
紀元前332年にはアレクサンドロス大王に征服された。その後、ギリシア系のプトレマ

KeyboardInterrupt: 

## 37. 名詞の出現頻度
コーパスにおける名詞の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 38. TF・IDF
日本に関する記事における名詞のTF・IDFスコアを求め、TF・IDFスコア上位20語とそのTF, IDF, TF・IDFを表示せよ。

## 39. Zipfの法則
コーパスにおける単語の出現頻度順位を横軸、その出現頻度を縦軸として、両対数グラフをプロットせよ。